# Cotações e indicadores fundamentalistas de ações
Extração de cotações históricas de ações da B3 com `yfinance` e de indicadores fundamentalistas com `fundamentus`, tratamento dos dados (formato longo, tipos numéricos, junção) e cálculo de LPA, VPA e do valor intrínseco de Graham, seguidos dos desafios do curso.

**Origem:** Alura — curso *Python e Power BI: análise de dados*.

**Dados:** obtidos ao vivo das APIs do Yahoo Finance (`yfinance`) e do site Fundamentus (`fundamentus`); não há arquivos para enviar. Os valores mudam conforme a data de execução.

**Índice**
1. Cotações com yfinance
2. Indicadores fundamentalistas com fundamentus
3. Transformação e tratamento
4. Desafios

In [ ]:
!pip install -q fundamentus yfinance

import fundamentus
import numpy as np
import pandas as pd
import yfinance as yf

## 1. Cotações com yfinance
O `yf.download` baixa as cotações diárias (abertura, fechamento, mínima, máxima e volume) de vários tickers de uma vez. Na B3 os tickers do Yahoo levam o sufixo `.SA`. O resultado tem colunas em dois níveis: (indicador, ticker).

In [ ]:
carteira_yf = ['ABEV3.SA', 'B3SA3.SA', 'GGBR4.SA', 'ITSA4.SA',
               'PETR4.SA', 'RENT3.SA', 'SUZB3.SA', 'VALE3.SA', 'WEGE3.SA']

df = yf.download(carteira_yf, start='2026-01-01', end='2026-08-05')
df

## 2. Indicadores fundamentalistas com fundamentus
O `fundamentus` lê os dados do site Fundamentus. Os tickers são usados sem o sufixo `.SA`.

### 2.1 Um papel

In [ ]:
weg = fundamentus.get_papel('WEGE3')
weg

### 2.2 Vários papéis com `get_papel`
Informações de cadastro e de balanço (cotação, faixa de 52 semanas, valor de mercado, número de ações, patrimônio líquido, receita e lucro).

In [ ]:
carteira_fund = ['ABEV3', 'B3SA3', 'AXIA3', 'GGBR4', 'ITSA4', 'PETR4', 'RENT3', 'SUZB3', 'VALE3', 'WEGE3']

ind = fundamentus.get_papel(carteira_fund)[['Setor', 'Cotacao', 'Min_52_sem', 'Max_52_sem', 'Valor_de_mercado',
                                            'Nro_Acoes', 'Patrim_Liq', 'Receita_Liquida_12m', 'Receita_Liquida_3m',
                                            'Lucro_Liquido_12m', 'Lucro_Liquido_3m']]
ind = ind.reset_index().rename(columns={'index': 'Ativo'})
ind

### 2.3 Indicadores de mercado com `get_resultado_raw`
Tabela com todos os papéis e seus múltiplos; filtramos a carteira e mantemos P/L, Dividend Yield, P/VP e ROE.

In [ ]:
ind_2 = fundamentus.get_resultado_raw().reset_index()
ind_2 = ind_2.query('papel in @carteira_fund')
ind_2 = ind_2[['papel', 'P/L', 'Div.Yield', 'P/VP', 'ROE']].reset_index(drop=True)
ind_2 = ind_2.rename(columns={'papel': 'Ativo', 'Div.Yield': 'DY'})
ind_2

## 3. Transformação e tratamento
### 3.1 Cotações em formato longo
O `stack(level=1)` leva os tickers das colunas para o índice (uma linha por data e ativo); depois o índice vira coluna e as colunas são reordenadas.

In [ ]:
cotacoes = df.stack(level=1).reset_index().rename(columns={'Ticker': 'Ativo'})
cotacoes = cotacoes[['Date', 'Ativo', 'Open', 'Close', 'Low', 'High', 'Volume']]
cotacoes

### 3.2 Tipos numéricos
O `get_papel` devolve os valores como texto; `pd.to_numeric(errors='coerce')` converte para número (o que não for número vira `NaN`).

In [ ]:
colunas = ['Cotacao', 'Min_52_sem', 'Max_52_sem', 'Valor_de_mercado', 'Nro_Acoes', 'Patrim_Liq',
           'Receita_Liquida_12m', 'Receita_Liquida_3m', 'Lucro_Liquido_12m', 'Lucro_Liquido_3m']
ind[colunas] = ind[colunas].apply(pd.to_numeric, errors='coerce')
ind.info()

### 3.3 Juntando as duas tabelas

In [ ]:
indicadores = pd.merge(ind, ind_2, on='Ativo')
indicadores

### 3.4 LPA e VPA
- **LPA** (lucro por ação) = lucro líquido dos últimos 12 meses / número de ações.
- **VPA** (valor patrimonial por ação) = patrimônio líquido / número de ações.

Com eles, P/L = cotação / LPA e P/VP = cotação / VPA; as colunas calculadas permitem conferir os valores trazidos pelo Fundamentus.

In [ ]:
indicadores['LPA'] = (indicadores['Lucro_Liquido_12m'] / indicadores['Nro_Acoes']).round(2)
indicadores['VPA'] = (indicadores['Patrim_Liq'] / indicadores['Nro_Acoes']).round(2)

# Conferência dos múltiplos
indicadores['P/L_calc'] = (indicadores['Cotacao'] / indicadores['LPA']).round(2)
indicadores['P/VP_calc'] = (indicadores['Cotacao'] / indicadores['VPA']).round(2)
indicadores[['Ativo', 'Cotacao', 'LPA', 'VPA', 'P/L', 'P/L_calc', 'P/VP', 'P/VP_calc']]

## 4. Desafios
Exercícios do curso aplicados a outra carteira. Os passos são os mesmos das seções 1 a 3, reunidos em um bloco por desafio.

### 4.1 Desafio 1 — Cotações de outra carteira (jan. a ago. de 2023), sem o volume

In [ ]:
tickers = ['ASAI3.SA', 'RAIL3.SA', 'EQTL3.SA', 'PRIO3.SA', 'RDOR3.SA', 'SBSP3.SA', 'VIVT3.SA']
df_desafio = yf.download(tickers, start='2023-01-01', end='2023-08-31')

cotacoes_desafio = df_desafio.stack(level=1).reset_index()
cotacoes_desafio = cotacoes_desafio[['Date', 'Ticker', 'Open', 'Close', 'High', 'Low']]
cotacoes_desafio

### 4.2 Desafio 2 — Indicadores de outra carteira, com dívida líquida e valor intrínseco de Graham
A fórmula de Graham estima o valor intrínseco de uma ação:

$$VI = \sqrt{22{,}5 \times LPA \times VPA}$$

Ela só faz sentido quando LPA e VPA são positivos; nos demais casos o resultado fica `NaN`.

In [ ]:
carteira_desafio = ['ASAI3', 'EQTL3', 'LREN3', 'PRIO3', 'RAIL3', 'RDOR3', 'SBSP3', 'VIVT3']

# Informações de cadastro e balanço, incluindo a dívida líquida
colunas_d = ['Cotacao', 'Min_52_sem', 'Max_52_sem', 'Nro_Acoes', 'Valor_de_mercado', 'Patrim_Liq',
             'Receita_Liquida_12m', 'Receita_Liquida_3m', 'Lucro_Liquido_12m', 'Lucro_Liquido_3m', 'Div_Liquida']
ind_d = fundamentus.get_papel(carteira_desafio)[['Setor'] + colunas_d]
ind_d = ind_d.reset_index().rename(columns={'index': 'Ativo'})
ind_d[colunas_d] = ind_d[colunas_d].apply(pd.to_numeric, errors='coerce')

# Múltiplos de mercado
ind_d2 = fundamentus.get_resultado_raw().reset_index()
ind_d2 = ind_d2.query('papel in @carteira_desafio')[['papel', 'P/L', 'P/VP', 'Div.Yield']].reset_index(drop=True)
ind_d2 = ind_d2.rename(columns={'papel': 'Ativo', 'Div.Yield': 'DY'})

# Junção, LPA, VPA e valor intrínseco de Graham
indicadores_desafio = pd.merge(ind_d, ind_d2, on='Ativo')
indicadores_desafio['LPA'] = (indicadores_desafio['Lucro_Liquido_12m'] / indicadores_desafio['Nro_Acoes']).round(2)
indicadores_desafio['VPA'] = (indicadores_desafio['Patrim_Liq'] / indicadores_desafio['Nro_Acoes']).round(2)

produto = 22.5 * indicadores_desafio['LPA'] * indicadores_desafio['VPA']
positivos = (indicadores_desafio['LPA'] > 0) & (indicadores_desafio['VPA'] > 0)
indicadores_desafio['VI_Graham'] = np.sqrt(produto.where(positivos)).round(2)
indicadores_desafio[['Ativo', 'Cotacao', 'LPA', 'VPA', 'VI_Graham', 'Div_Liquida', 'P/L', 'P/VP', 'DY']]